# 🚀 EINSOF / POLYDIM — SOTA Fine-Tuning Suite (Unsloth QLoRA) — V020
**Autor:** Ariel García Traba (Independent Research / EinsofOS)  
**Corpus Canónico:** 1.736 Teoremas de PostgreSQL 16 (`polydim_vec_db`) + 13 Leyes de Silicio  
**Compatibilidad:** Google Colab Free (**NVIDIA Tesla T4 16GB**) y Colab Plus / Pro (**NVIDIA A100 40GB / 80GB**)

---

### 🎯 Modelos SOTA Soportados
1. **`DeepSeek-R1-Distill-Qwen-14B`:** Rey del razonamiento formal y matemáticas. **(Apto Colab Free T4 16GB)**
2. **`Qwen2.5-Coder-32B-Instruct`:** SOTA mundial en código C++20, Rust, ABI y FFI. **(Apto Colab Plus A100 40GB/80GB)**
3. **`Qwen2.5-72B-Instruct`:** Copiloto gigante formal de EinsofOS — Cumplimiento estricto **Regla 45**. **(Apto A100 80GB)**
4. **`Qwen2.5-Coder-7B-Instruct`:** Opción ligera para pruebas ultrarrápidas de 10 minutos en T4.

> [!IMPORTANT]
> Este notebook es **100% autocontenido**: incluye el dataset comprimido de los 1.736 teoremas embebido en código. No requiere subir archivos manualmente para iniciar el entrenamiento.


In [ ]:
# [1] Instalación de dependencias aceleradas con Unsloth (2x-5x más rápido, 70% menos VRAM)
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets

import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# [2] Selector de Modelo y Configuración según Hardware Detectado

# Opciones disponibles:
# 1. 'deepseek-14b' -> DeepSeek-R1-Distill-Qwen-14B (Óptimo para T4 Free y A100)
# 2. 'qwen-32b-coder' -> Qwen2.5-Coder-32B-Instruct (Para A100 40GB/80GB)
# 3. 'qwen-72b' -> Qwen2.5-72B-Instruct (Para A100 80GB - Regla 45 Einsof)
# 4. 'qwen-7b-coder' -> Qwen2.5-Coder-7B-Instruct (Ligero para T4 Free)

MODEL_CHOICE = "deepseek-14b"  # @param ["deepseek-14b", "qwen-32b-coder", "qwen-72b", "qwen-7b-coder"]

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0.0

MODEL_MAP = {
    "deepseek-14b": ("unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit", 2048, 14.7),
    "qwen-32b-coder": ("unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit", 4096 if vram_gb > 30 else 2048, 32.5),
    "qwen-72b": ("unsloth/Qwen2.5-72B-Instruct-bnb-4bit", 4096 if vram_gb > 70 else 2048, 72.7),
    "qwen-7b-coder": ("unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", 2048, 7.6),
}

model_id, max_seq_length, param_count = MODEL_MAP[MODEL_CHOICE]
print(f"=================================================================")
print(f"  CONFIGURACIÓN SELECCIONADA:")
print(f"  Modelo:          {model_id} ({param_count}B params)")
print(f"  Max Seq Length:  {max_seq_length}")
print(f"  GPU Activa:      {gpu_name} ({vram_gb:.2f} GB VRAM)")
print(f"=================================================================")

if vram_gb < 20 and param_count > 20:
    print(f"⚠️ ADVERTENCIA: {model_id} requiere mínimo 24GB de VRAM. Se recomienda usar 'deepseek-14b' o 'qwen-7b-coder' en T4 Free.")


In [ ]:
# [3] Carga del Modelo en 4-Bit Cuantizado y Configuración LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = None,          # Auto-detecta FP16 para T4 o BF16 para A100
    load_in_4bit = True,   # Cuantización de alta fidelidad 4-bit
)

# Configurar adaptadores LoRA sobre todas las capas lineales de atención y MLP
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # Rango de adaptación LoRA
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,      # Optimizado a 0 por Unsloth
    bias = "none",         # Optimizado a none
    use_gradient_checkpointing = "unsloth", # Ahorro de VRAM para secuencias largas
    random_state = 20261018,
)

vram_used = torch.cuda.memory_allocated() / (1024**3)
print(f">>> Modelo cargado exitosamente en VRAM: {vram_used:.2f} GB utilizados.")


In [ ]:
# [4] Descompresión y Preparación del Dataset Formal Einsof (1.736 Teoremas)
import gzip
import base64
import json
from datasets import Dataset

# Dataset canónico embebido (comprimido en base64)
B64_DATASET = (
    "..."
)

print("Descomprimiendo einsof_dataset_training.jsonl embebido...")
raw_decompressed = gzip.decompress(base64.b64decode(B64_DATASET))
dataset_lines = raw_decompressed.decode('utf-8').strip().split('\n')
print(f"Total de muestras canónicas de teoremas cargadas: {len(dataset_lines)}")

data_records = [json.loads(line) for line in dataset_lines]

# Formatear según el Chat Template estándar de Qwen / DeepSeek
formatted_texts = []
for item in data_records:
    messages = item["messages"]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    formatted_texts.append(formatted)

train_dataset = Dataset.from_dict({"text": formatted_texts})
print(f">>> Dataset preparado con éxito: {len(train_dataset)} ejemplos listos para entrenamiento.")
print(f"Ejemplo de muestra 0:\n{formatted_texts[0][:250]}...")


In [ ]:
# [5] Ejecución del Entrenamiento con SFTTrainer y Aceleración Unsloth
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Ajuste dinámico de batch según GPU
batch_size = 2 if vram_gb < 30 else 4
grad_accum = 4 if vram_gb < 30 else 2

training_args = TrainingArguments(
    per_device_train_batch_size = batch_size,
    gradient_accumulation_steps = grad_accum,
    warmup_steps = 10,
    max_steps = 120,            # 120 steps proporcionan una inyección sólida (~15 min en T4, ~6 min en A100)
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 20261018,
    output_dir = "einsof_outputs",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Puede activarse si se desea entrenar múltiples secuencias cortas en una
    args = training_args,
)

print("=================================================================")
print("  INICIANDO ENTRENAMIENTO LORA / SFT EINSOF-V020")
print("=================================================================")
trainer_stats = trainer.train()
print("\n>>> ENTRENAMIENTO COMPLETADO EXITOSAMENTE.")
print(f"Tiempo total de entrenamiento: {trainer_stats.metrics['train_runtime']:.2f} segundos")


In [ ]:
# [6] Batería de Inferencia y Verificación de Teoremas Post-Fine-Tuning
FastLanguageModel.for_inference(model) # Activa modo de inferencia acelerado nativo

test_prompts = [
    "Explique el Teorema 9.5 (V44.0) de sincronización Seqlock par/impar en memoria compartida y cómo erradica las lecturas desgarradas.",
    "Defina la retracción polar de orden 5 de Newton-Schulz sobre la variedad de Stiefel St(k, d) y su radio de atracción espectral.",
    "¿Cómo opera el broker de EinsofOS para aislar procesos con JobObject en Windows y evitar fugas de handles?"
]

for i, p in enumerate(test_prompts, 1):
    print(f"\n==================== TEST {i} ====================")
    print(f"PROMPT: {p}\n")
    
    messages = [
        {"role": "system", "content": "Sos EINSOF-Qwen, el modelo fundacional de EinsofOS y Einsof Language. Operas con rigor formal en variedades de Stiefel y silicio C++20."},
        {"role": "user", "content": p}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 256,
        use_cache = True,
        temperature = 0.2,
        do_sample = True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"RESPUESTA DEL MODELO EINSOF:\n{response}\n")


In [ ]:
# [7] Exportación de Adaptadores LoRA y Publicación a Hugging Face / GGUF
import os

SAVE_DIR = f"einsof_lora_{MODEL_CHOICE}_v020"
print(f"Guardando adaptadores LoRA en '{SAVE_DIR}'...")
model.save_pretrained_merged(SAVE_DIR, tokenizer, save_method="lora")
print(f">>> Adaptadores guardados exitosamente en carpeta local: {SAVE_DIR}")

# Opcional: Subida directa a Hugging Face Hub
# Descomenta las siguientes líneas e introduce tu HF_TOKEN para publicar:
# HF_REPO = "tu_usuario/einsof-lora-v020"
# model.push_to_hub_merged(HF_REPO, tokenizer, save_method="lora", token="tu_hf_token")

# Opcional: Exportar a GGUF (para Ollama en CPU / Aladín):
# model.save_pretrained_gguf("einsof_gguf_v020", tokenizer, quantization_method = "q4_k_m")
